# Reshaping Data in Pandas

Explore wide and long formats, melt, pivot, pivot_table, stack, unstack, and crosstab.

In [1]:
import pandas as pd

wide = pd.DataFrame({'student': ['Asha', 'Ravi'], 'math': [90, 78], 'science': [85, 88]})
wide

,student,math,science
0,Asha,90,85
1,Ravi,78,88


## 1. Wide to long with melt()

In [2]:
long = wide.melt(id_vars='student', var_name='subject', value_name='score')
long

,student,subject,score
0,Asha,math,90
1,Ravi,math,78
2,Asha,science,85
3,Ravi,science,88


## 2. Long to wide with pivot()

In [3]:
long.pivot(index='student', columns='subject', values='score').reset_index()

subject,student,math,science
0,Asha,90,85
1,Ravi,78,88


## 3. pivot() requires unique combinations

In [4]:
repeated = pd.DataFrame({'region': ['East', 'East', 'West'], 'product': ['A', 'A', 'A'], 'revenue': [100, 150, 200]})
print('Duplicate region/product pairs:')
display(repeated[repeated.duplicated(['region', 'product'], keep=False)])
print('Use pivot_table when aggregation is intended:')
display(repeated.pivot_table(index='region', columns='product', values='revenue', aggfunc='sum'))

Duplicate region/product pairs:


,region,product,revenue
0,East,A,100
1,East,A,150


Use pivot_table when aggregation is intended:


product,A
region,
East,250
West,200


## 4. Pivot tables and missing combinations

In [5]:
repeated.pivot_table(index='region', columns='product', values='revenue', aggfunc='sum', fill_value=0, margins=True)

product,A,All
region,,
East,250,250
West,200,200
All,450,450


## 5. Crosstab and normalization

In [6]:
transactions = pd.DataFrame({'region': ['East', 'East', 'West', 'West', 'West'], 'product': ['A', 'B', 'A', 'A', 'B']})
print('Counts:')
display(pd.crosstab(transactions['region'], transactions['product']))
print('Row proportions:')
display(pd.crosstab(transactions['region'], transactions['product'], normalize='index'))

Counts:


product,A,B
region,,
East,1,1
West,2,1


Row proportions:


product,A,B
region,,
East,0.500000,0.500000
West,0.666667,0.333333


## 6. Stack and unstack

In [7]:
indexed = wide.set_index('student')
print('Stacked Series:')
display(indexed.stack())
print('Unstack back to columns:')
display(indexed.stack().unstack())

Stacked Series:


student         
Asha     math       90
         science    85
Ravi     math       78
         science    88
dtype: int64

Unstack back to columns:


,math,science
student,,
Asha,90,85
Ravi,78,88


## 7. End-to-end reshape and summarize

In [8]:
monthly_sales = pd.DataFrame({'region': ['East', 'West'], 'Jan': [100, 200], 'Feb': [150, 250], 'Mar': [120, 300]})
long_sales = monthly_sales.melt(id_vars='region', var_name='month', value_name='sales')
summary = long_sales.groupby(['region', 'month'], as_index=False).agg(total_sales=('sales', 'sum'))
report = summary.pivot(index='region', columns='month', values='total_sales')
print('Long data:')
display(long_sales)
print('Report:')
display(report)
assert long_sales['sales'].sum() == monthly_sales[['Jan', 'Feb', 'Mar']].to_numpy().sum()

Long data:


,region,month,sales
0,East,Jan,100
1,West,Jan,200
2,East,Feb,150
3,West,Feb,250
4,East,Mar,120
5,West,Mar,300


Report:


month,Feb,Jan,Mar
region,,,
East,150,100,120
West,250,200,300


## Key takeaways

- Use `melt()` to convert wide data to long format.
- Use `pivot()` for unique combinations and `pivot_table()` when aggregation is required.
- Use `crosstab()` for category frequencies and proportions.
- Verify that totals and intended row granularity are preserved.